# Recursive Language Models for Poker Decision-Making

**STAT 4830 Final Project · Aadithya Srinivasan, Aryan Arora, Aarav M.**

---

This notebook is a self-contained walkthrough of the entire project:

| Section | What you'll see |
|---|---|
| 1. Setup | Install deps, clone repo, verify GPU |
| 2. The Task | What a poker episode looks like |
| 3. The Architecture | How the agent writes and runs Python code |
| 4. Live Demo Traces | Three real agent rollouts with full code |
| 5. Training Story | How performance evolved across all experiments |
| 6. Quantitative Eval | Agent comparison table |
| 7. What Went Wrong | Reward hacking and the REPL fix |
| 8. Next Steps | Slumbot evaluation + future work |

**Runtime:** ~10 minutes on a free-tier Colab T4 GPU (inference only).  
For full BC + RL training (~90 min) see `scripts/primeintellect/README.md`.

> **Quick start:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

## 1. Setup

In [ ]:
# Install all dependencies
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes matplotlib numpy

import os
if not os.path.exists('STAT-4830-RL-project'):
    !git clone https://github.com/aryanarora236/STAT-4830-RL-project.git
os.chdir('STAT-4830-RL-project')
!git pull --ff-only

import sys
sys.path.insert(0, '.')
print('Repo ready.')

In [ ]:
import torch
assert torch.cuda.is_available(), "Go to Runtime > Change runtime type > T4 GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}  ({gpu.total_memory / 1e9:.1f} GB)")

# Run tests to confirm the codebase is intact
!python -m pytest tests/ -q --tb=line 2>&1 | tail -5

## 2. The Task — What a Poker Episode Looks Like

Each episode is a tuple `(context, question, answer*)` where:

- **Context** (~4,300 characters): the current hand state (hole cards, board, positions, stacks, pot odds) **plus** 15 previous hand records from which opponent tendencies can be inferred.
- **Question**: a fixed prompt asking for the correct action.
- **Answer\***: the ground-truth action produced by the heuristic bot (`fold / check / call $X / raise $X`).

The key design choice: **the context lives in a Python variable** (`CONTEXT`), not in the prompt. The agent must write code to parse it — which forces real reasoning rather than pattern matching.

In [ ]:
import random
from src.poker.tasks import generate_poker_task, generate_preflop_task, generate_postflop_task
from src.poker.rewards import parse_action

random.seed(4830)
context, question, correct = generate_poker_task()

print("=" * 70)
print("FULL CONTEXT (trimmed to 1200 chars):")
print("=" * 70)
print(context[:1200])
print("...")
print(f"\n[Total context length: {len(context)} characters]")
print("\n" + "=" * 70)
print("QUESTION:")
print("=" * 70)
print(question)
print("\n" + "=" * 70)
print(f"CORRECT ANSWER (from heuristic teacher): {correct!r}")
print("=" * 70)

## 3. The Architecture — Agent Writes Python, Sandbox Runs It

```
Poker Task (context, question)
          │
          ▼
   ┌─────────────┐
   │   LLM Agent │  writes Python code
   └──────┬──────┘
          │  code string
          ▼
   ┌─────────────┐
   │ Sandboxed   │  executes with CONTEXT global
   │ Python REPL │  (5-sec timeout, whitelist-only)
   └──────┬──────┘
          │  last printed line
          ▼
   Predicted action:  fold / check / call $X / raise $X
```

The RLM pattern: instead of trying to attend over 4k characters of poker history, the model *writes code* that parses it. This is the key technical contribution.

**Training:** two phases.
1. **Behavior Cloning (BC):** fine-tune Qwen2.5-Coder-1.5B on 500 expert (heuristic) `(prompt, code)` pairs using LoRA. Teaches the retrieve→compute→decide pattern.
2. **REINFORCE:** starting from the BC checkpoint, run policy gradient updates on live rollouts. Reward = action-type match vs. teacher (1.0 / 0.3 / 0.0).

In [ ]:
# Show the heuristic's ideal code for the task above — the target BC teaches the LLM to write
from src.poker.agents import PokerHeuristicAgent
from src.utils import safe_execute_code

heuristic = PokerHeuristicAgent()
heur_pred, heur_trace = heuristic.run_episode(context, question, correct)

print("=" * 70)
print("HEURISTIC AGENT — ideal retrieve→compute→decide code:")
print("=" * 70)
heur_codes = [e.get('code', '') for e in heur_trace if 'code' in e]
if heur_codes:
    print(heur_codes[-1][:1500])
print(f"\nHeuristic predicted: {heur_pred!r}  |  Correct: {correct!r}")

## 4. Live Demo Traces

We load the best RL checkpoint (iteration 105 of the 120-iteration run, EMA accuracy 42.3%) and run three live poker scenarios. For each we show:
- The game state
- The exact Python code the model wrote
- The sandbox output
- Whether the action matched the teacher

In [ ]:
# Load model — try the best available RL checkpoint, fall back to zero-shot.
from src.training import load_model

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

# Ordered by held-out accuracy (see docs/results/analysis_all_experiments_20260423/experiment_summary.csv).
# Both paths point at LoRA adapters trained on Qwen2.5-Coder-1.5B-Instruct.
CKPT_CANDIDATES = [
    ('docs/results/poker_rl_expA_partial_20260422/poker_rl_expA_evalselect_20260421_long/best_by_eval',
     'RL-finetuned (Exp A best_by_eval, 25% held-out acc)'),
    ('docs/results/poker_rl_expB_mixed20_20260422/poker_rl_expB_mixed20_20260422/best_by_eval',
     'RL-finetuned (Exp B best_by_eval, 20% held-out acc)'),
    ('checkpoints/poker_rl',  # legacy path if anyone has it downloaded locally
     'RL-finetuned (local checkpoint)'),
]

model = tokenizer = None
MODE = 'zero-shot'
for path, label in CKPT_CANDIDATES:
    if os.path.exists(os.path.join(path, 'adapter_config.json')):
        print(f'Loading {label}\n  from: {path}')
        model, tokenizer = load_model(path, load_in_4bit=True)
        MODE = label
        break

if model is None:
    print('No RL checkpoint found — loading base model (zero-shot demo).')
    print('Expected one of:')
    for path, _ in CKPT_CANDIDATES:
        print(f'  - {path}/adapter_config.json')
    model, tokenizer = load_model(MODEL_ID, load_in_4bit=True, lora_r=16)

model.eval()
print(f'\nReady — {MODE}.')

In [ ]:
from src.poker.agents import PokerLocalLLMAgent

llm_agent = PokerLocalLLMAgent(
    model=model, tokenizer=tokenizer,
    name=f'Qwen1.5B-{MODE}', max_steps=3, temperature=0.1,
)

random.seed(4830)

scenarios = [
    ('Preflop — read opponent VPIP',  generate_preflop_task),
    ('Postflop — pot odds + board',   generate_postflop_task),
    ('Mixed streets',                  generate_poker_task),
]

for label, gen in scenarios:
    print()
    print('=' * 72)
    print(f'SCENARIO: {label}')
    print('=' * 72)

    ctx, q, ans = gen()
    print('CONTEXT (first 500 chars):')
    print(ctx[:500].rstrip() + '...')
    print()

    # Run LLM agent
    llm_pred, llm_trace = llm_agent.run_episode(ctx, q, ans)
    codes = [e.get('code', '') for e in llm_trace if 'code' in e]

    print('--- LLM AGENT ---')
    if codes:
        print('Code written (first 600 chars):')
        print(codes[-1][:600].rstrip())
        exec_results = [e.get('exec_result', {}) for e in llm_trace if 'exec_result' in e]
        if exec_results:
            last_exec = exec_results[-1]
            status = 'OK' if last_exec.get('ok') else 'ERROR'
            stdout = last_exec.get('stdout', '').strip()
            stderr = last_exec.get('stderr', '').strip()[:200]
            print(f'\nExecution: {status}')
            if stdout:
                print(f'Stdout: {stdout!r}')
            if stderr:
                print(f'Stderr: {stderr!r}')
    else:
        print('(No code extracted — fallback to direct text parse)')

    match = parse_action(llm_pred)[0] == parse_action(ans)[0]
    print(f'\nPredicted: {llm_pred!r}  |  Correct: {ans!r}  |  Match: {"✓" if match else "✗"}')
    print()

### 4.1 Pre-recorded Sample Traces (from the shaped-reward RL run)

The three examples below are verbatim from the shaped-reward Modal run (Experiment v1, April 23). They illustrate the spectrum from reward hacking to genuine reasoning.

---

**Sample 1 — The shortcut (reward hack):** One-line fold inside a code fence. Earns the code-bonus without any reasoning.

```python
print("fold")
```

---

**Sample 2 — The attempt:** Model scans game history for opponent stats. Gets truncated mid-loop and crashes on an undefined name.

```python
import re
lines = CONTEXT.split('\n')
for i, line in enumerate(lines):
    if '=== PREVIOUS HANDS' in line:
        history_start = i
        break
stats = {}
# (truncated — crashes on undefined name 'stats' in compute step)
```

---

**Sample 3 — Real strategic reasoning:** Folds marginal hands against tight players, raises with flush draws. This is the behavior the RLM framework is designed for.

```python
# Tight (low VPIP): respect bets, fold marginals
elif opp_stats['VPIP'] <= 0.4:
    if strength['flush_draw']:
        print("call {}".format(to_call))
    else:
        print("raise {}".format(max(10, to_call * 1.5)))
```

---

The progression from Sample 1 → Sample 3 is the learning signal. The reward-shaping fix (Section 7) makes Sample 1 less rewarding so the policy pushes toward Samples 2 and 3.

### 4.2 Visualize the RLM's decisions

The text traces above show *what* the model decided. This section shows *what it saw* — a poker-table render of three fresh tasks with the model's chosen action highlighted, color-coded against the heuristic teacher's answer. Useful when reviewing whether the model picked a reasonable action in context, and good for slides.

Three fresh tasks are sampled from the same generator §4.1 used, so the visualisation matches the model's training distribution (6-max NLHE, $1/$2 blinds).


In [ ]:
# _RLM_RENDERER_MARKER — poker-table visualisation with matplotlib.
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Disable mathtext so $ signs render literally instead of triggering math mode.
mpl.rcParams["text.parse_math"] = False

_SUIT_SYM = {"s": "♠", "h": "♥", "d": "♦", "c": "♣"}


def _draw_card(ax, x, y, w, h, card_str=None):
    """Draw one playing card at (x, y), size (w, h). None = facedown back."""
    if not card_str or card_str.startswith("?"):
        rect = FancyBboxPatch((x, y), w, h,
                              boxstyle="round,pad=0.02,rounding_size=0.05",
                              facecolor="#1e3a5f", edgecolor="black", linewidth=1.2)
        ax.add_patch(rect)
        for i in range(3):
            for j in range(4):
                ax.text(x + w * (0.2 + i * 0.3), y + h * (0.15 + j * 0.25),
                        "♦", ha="center", va="center", fontsize=8, color="#3a5d8f")
        return
    rank, suit = card_str[0].upper(), card_str[1].lower()
    sym = _SUIT_SYM.get(suit, "?")
    color = "#c41e3a" if suit in "hd" else "black"
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle="round,pad=0.02,rounding_size=0.05",
                          facecolor="white", edgecolor="black", linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x + w * 0.18, y + h * 0.82, rank, ha="center", va="center",
            fontsize=13, color=color, weight="bold")
    ax.text(x + w * 0.18, y + h * 0.62, sym, ha="center", va="center",
            fontsize=10, color=color)
    ax.text(x + w * 0.5, y + h * 0.5, sym, ha="center", va="center",
            fontsize=22, color=color)
    ax.text(x + w * 0.82, y + h * 0.18, rank, ha="center", va="center",
            fontsize=10, color=color, weight="bold")


_STREET_NAMES = ["Preflop", "Flop", "Turn", "River"]


def render_hand_state(hole_cards, board, pot, to_call, our_stack, opp_stack,
                      position, opp_label="Opponent",
                      predicted_action=None, correct_action=None,
                      title="", action_history=None):
    """Render a single hand state as a matplotlib Figure (slide-friendly)."""
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.set_xlim(0, 11); ax.set_ylim(0, 7)
    ax.set_aspect("equal"); ax.axis("off")

    if title:
        ax.text(5.5, 6.65, title, ha="center", va="center",
                fontsize=13, weight="bold", color="#222")

    # Felt
    ax.add_patch(FancyBboxPatch((0.4, 0.7), 10.2, 5.65,
                                boxstyle="round,pad=0.05,rounding_size=0.4",
                                facecolor="#1f6e3f", edgecolor="#3b1f0d", linewidth=3))
    ax.add_patch(FancyBboxPatch((0.7, 0.95), 9.6, 5.15,
                                boxstyle="round,pad=0.05,rounding_size=0.35",
                                facecolor="none", edgecolor="#0d4926", linewidth=1.5))

    opp_seat = "BTN" if position == "BB" else "BB"
    ax.text(5.5, 5.85, f"{opp_label}  ({opp_seat})  —  ${opp_stack:,}",
            ha="center", va="center", fontsize=11, color="white", weight="bold")
    _draw_card(ax, 4.9, 4.6, 0.55, 0.85, None)
    _draw_card(ax, 5.5, 4.6, 0.55, 0.85, None)

    # Board (5 slots)
    n_board = len(board)
    card_w, card_h, gap = 0.65, 0.95, 0.1
    total_w = 5 * card_w + 4 * gap
    start_x = 5.5 - total_w / 2
    for i in range(5):
        x = start_x + i * (card_w + gap)
        if i < n_board:
            _draw_card(ax, x, 3.0, card_w, card_h, board[i])
        else:
            ax.add_patch(FancyBboxPatch((x, 3.0), card_w, card_h,
                                        boxstyle="round,pad=0.02,rounding_size=0.05",
                                        facecolor="none", edgecolor="#155228",
                                        linewidth=1, linestyle="--"))

    # Pot
    pot_h = 0.7 if to_call else 0.45
    ax.add_patch(FancyBboxPatch((4.25, 2.15), 2.5, pot_h,
                                boxstyle="round,pad=0.05,rounding_size=0.1",
                                facecolor="#fff5d9", edgecolor="#8a6d1c", linewidth=1.5))
    if to_call:
        odds = to_call / (pot + to_call) * 100
        ax.text(5.5, 2.65, f"Pot: ${pot:,}", ha="center", va="center",
                fontsize=12, weight="bold", color="#5a4a10")
        ax.text(5.5, 2.30, f"To call ${to_call:,}  ({odds:.0f}% odds)",
                ha="center", va="center", fontsize=10, color="#5a4a10")
    else:
        ax.text(5.5, 2.375, f"Pot: ${pot:,}", ha="center", va="center",
                fontsize=12, weight="bold", color="#5a4a10")

    # Hero
    _draw_card(ax, 4.9, 0.95, 0.55, 0.85, hole_cards[0])
    _draw_card(ax, 5.5, 0.95, 0.55, 0.85, hole_cards[1])
    ax.text(5.5, 0.78, f"You  ({position})  —  ${our_stack:,}",
            ha="center", va="center", fontsize=11, color="white", weight="bold")

    # Action history side panel
    if action_history:
        text = "\n".join(action_history[-6:])
        ax.text(7.1, 5.3, "History",
                ha="left", va="top", fontsize=9, color="#cbe5d3", weight="bold")
        ax.text(7.1, 5.1, text,
                ha="left", va="top", fontsize=8.5, color="white", family="monospace")

    # Decision banner
    if predicted_action:
        text = f"RLM decision: {predicted_action}"
        bg = "#c41e3a"
        if correct_action is not None:
            pred_a = predicted_action.strip().lower().split()[0] if predicted_action.strip() else ""
            corr_a = correct_action.strip().lower().split()[0] if correct_action.strip() else ""
            if pred_a == corr_a:
                text += "    [agrees with heuristic]"
                bg = "#1a7a3a"
            else:
                text += f"    [heuristic: {correct_action}]"
                bg = "#a02929"
        ax.text(5.5, 0.3, text, ha="center", va="center",
                fontsize=13, weight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.35", facecolor=bg,
                          edgecolor="black", linewidth=1.5))

    plt.tight_layout()
    return fig


# Lightweight context-string parser: pulls structured fields out of the
# textual prompt the model receives, so the renderer can show what the model
# saw. Stable against changes in non-card sections.
_RE_HAND     = re.compile(r"Your Hand:\s*(\S+)\s+(\S+)")
_RE_POSITION = re.compile(r"Your Position:\s*(\w+)")
_RE_STACK    = re.compile(r"Your Stack:\s*\$([\d.]+)")
_RE_POT      = re.compile(r"Pot:\s*\$([\d.]+)")
_RE_TO_CALL  = re.compile(r"To Call:\s*\$([\d.]+)")
_RE_BOARD    = re.compile(r"Community Cards:\s*([^(]+?)\s*\(")
_RE_OPP_AMTS = re.compile(r"^  \w+:\s*\$([\d.]+)", re.MULTILINE)


def parse_context_for_render(context: str) -> dict:
    """Pull hole_cards, board, pot, to_call, position, stacks from the prompt."""
    out = {}
    m = _RE_HAND.search(context)
    out["hole_cards"] = [m.group(1), m.group(2)] if m else ["??", "??"]
    m = _RE_POSITION.search(context)
    out["position"] = m.group(1) if m else "??"
    m = _RE_STACK.search(context)
    out["our_stack"] = int(float(m.group(1))) if m else 0
    m = _RE_POT.search(context)
    out["pot"] = int(float(m.group(1))) if m else 0
    m = _RE_TO_CALL.search(context)
    out["to_call"] = int(float(m.group(1))) if m else 0
    m = _RE_BOARD.search(context)
    if m:
        out["board"] = [c for c in m.group(1).strip().split() if c]
    else:
        out["board"] = []
    opp_amts = [int(float(x)) for x in _RE_OPP_AMTS.findall(context)]
    out["opp_stack"] = max(opp_amts) if opp_amts else 200
    return out


print("Renderer ready: render_hand_state(...) and parse_context_for_render(ctx).")


In [ ]:
# _VIZ_DEMO_MARKER — render three live RLM decisions as poker-table figures.
import random
import os
from src.poker.tasks import generate_preflop_task, generate_postflop_task, generate_poker_task

random.seed(4830)
scenarios = [
    ("Preflop decision",  generate_preflop_task),
    ("Postflop decision", generate_postflop_task),
    ("Mid-hand decision", generate_poker_task),
]

os.makedirs("figures", exist_ok=True)
rendered_paths = []

for label, gen in scenarios:
    ctx, question, correct = gen()
    predicted, _trace = llm_agent.run_episode(haystack=ctx, question=question, correct_answer=correct)

    state = parse_context_for_render(ctx)
    fig = render_hand_state(
        hole_cards=state["hole_cards"],
        board=state["board"],
        pot=state["pot"],
        to_call=state["to_call"],
        our_stack=state["our_stack"],
        opp_stack=state["opp_stack"],
        position=state["position"],
        opp_label="Heuristic teacher",
        predicted_action=predicted,
        correct_action=correct,
        title=label,
    )
    out_path = f"figures/rlm_demo_{label.lower().replace(' ', '_')}.png"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    rendered_paths.append(out_path)
    plt.show()

print(f"\nRendered {len(rendered_paths)} figures:")
for p in rendered_paths:
    print(f"  - {p}")


## 5. Training Story — How Performance Evolved

We ran five experiments, each motivated by what the previous one revealed.

| Experiment | Key change | Best held-out acc |
|---|---|---|
| Proof of concept (3–10 iters) | Verify pipeline works | n/a (no held-out eval) |
| Long run (120 iters) | Scale iterations | n/a (no held-out eval; EMA 42.3%) |
| Experiment A (50 iters) | Add held-out eval, bigger batch | **25.0%** @ iter 35 |
| Experiment B (100 iters) | Longer run, smaller batch | **20.0%** (flat) |
| Fast Mixed run (20 iters) | Shorter, earlier eval | **40.0%** @ iter 10 |
| Shaped Reward (20 iters) | Reward code use explicitly | **33.3%** @ iter 5, 10 |

Key insight: **the biggest improvement was methodology**, not hyperparameters. Moving from training-only metrics to held-out evaluation with best-checkpoint selection is what unlocked the real gains.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Held-out eval data (from eval_leaderboard.csv files) ──────────────────
exp_a_iters  = [5, 10, 15, 20, 25, 30, 35, 40, 45]
exp_a_acc    = [0.25, 0.25, 0.233, 0.233, 0.25, 0.233, 0.25, 0.25, 0.233]

exp_b_iters  = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
exp_b_acc    = [0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20]

fast_iters   = [10, 20]
fast_acc     = [0.40, 0.25]

shaped_iters = [5, 10]
shaped_acc   = [0.333, 0.333]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Held-out accuracy over time ─────────────────────────────────────
ax = axes[0]
ax.plot(exp_a_iters,  exp_a_acc,  'o-', color='#2980b9', linewidth=2, markersize=6, label='Exp A (batch 6, eval every 5)')
ax.plot(exp_b_iters,  exp_b_acc,  's--', color='#e74c3c', linewidth=2, markersize=6, label='Exp B (batch 4, 100 iters)')
ax.plot(fast_iters,   fast_acc,   '^-', color='#27ae60', linewidth=2, markersize=8, label='Fast Mixed (batch 4, 20 iters)')
ax.plot(shaped_iters, shaped_acc, 'D-', color='#8e44ad', linewidth=2, markersize=7, label='Shaped Reward (code bonus)')
ax.axhline(0.20, color='#e74c3c', linestyle=':', alpha=0.4, linewidth=1)
ax.set_xlabel('RL Iteration', fontsize=12)
ax.set_ylabel('Held-out Accuracy', fontsize=12)
ax.set_title('Held-out Eval Accuracy per Experiment', fontsize=13, fontweight='bold')
ax.set_ylim(-0.02, 0.55)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.3)

# ── Right: Best held-out per experiment bar chart ─────────────────────────
ax2 = axes[1]
labels = ['Exp B\n(100 iters)', 'Exp A\n(50 iters)', 'Fast Mixed\n(20 iters)', 'Shaped\nReward']
accs   = [0.20, 0.25, 0.40, 0.333]
colors = ['#e74c3c', '#2980b9', '#27ae60', '#8e44ad']

bars = ax2.bar(labels, accs, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
             f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.axhline(0.20, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='Exp B baseline')
ax2.set_ylabel('Best Held-out Accuracy', fontsize=12)
ax2.set_title('Best Held-out Accuracy per Experiment', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 0.55)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/held_out_eval_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to figures/held_out_eval_summary.png')

In [ ]:
# Load and display the existing training curves from the long RL run (if present)
from IPython.display import Image, display

curve_candidates = [
    'docs/results/poker_rl_long_simple_120iters_20260420/training_curves.png',
    'docs/results/poker_rl_expA_partial_20260422/poker_rl_expA_evalselect_20260421_long/training_curves.png',
    'docs/results/poker_rl_expB_mixed20_20260422/poker_rl_expB_mixed20_20260422/training_curves.png',
]

for path in curve_candidates:
    if os.path.exists(path):
        run_name = path.split('/')[-2]
        print(f'Training curves: {run_name}')
        display(Image(path, width=800))
        print()

## 6. Quantitative Evaluation — Agent Comparison

All agents evaluated on the **same fixed task set**, same seed, same scoring function.

In [ ]:
from src.poker.evaluation import PokerEvaluationFramework

random.seed(2026)
agents = [llm_agent, heuristic]

print('Running quantitative eval (30 episodes)...')
eval_framework = PokerEvaluationFramework(
    agents=agents,
    task_generator=generate_poker_task,
    num_episodes=30,
)
eval_framework.run_evaluation()
eval_framework.display_results()
print()
eval_framework.display_confusion_matrix(llm_agent.name)

In [ ]:
# Published experiment summary table
import pandas as pd
from IPython.display import display

summary_data = {
    'Agent': [
        'Zero-shot Qwen-7B (HF API)',
        'Exp B best_by_eval (unshaped)',
        'Exp A best_by_eval (iter 35)',
        'Fast Mixed best (iter 10)',
        'Shaped Reward (iter 5/10)',
        'Heuristic teacher',
    ],
    'Held-out Acc': ['8.0%', '20.0%', '25.0%', '40.0%', '33.3%', '100.0%'],
    'Held-out Reward': ['0.092', '0.200', '0.285', '0.460', '0.333', '1.000'],
    'N episodes': [25, 20, 60, 20, 15, 25],
    'Note': [
        'No fine-tuning',
        'Flat across 100 iters',
        'First gen signal',
        'Best overall',
        '+13pp vs Exp B',
        'Oracle ceiling',
    ]
}
df = pd.DataFrame(summary_data)
display(df.to_string(index=False))

## 7. What Went Wrong — Reward Hacking and the REPL Fix

The most important empirical finding of this project is a **failure mode**.

### The problem: the policy stopped using the REPL

In Experiment B, we logged `real_code` (rollouts where the model wrote actual Python) vs `wrapped_action_code` (rollouts where it just printed one word like `fold`):

| Metric | Exp B average (per batch of 4) |
|---|---|
| `real_code` | **0 / 4** |
| `wrapped_action_code` | **4 / 4** |

Every single rollout just emitted `fold` or `call` as a text string. The Python REPL we built might as well not exist. This is reward hacking: the type-match reward fires ~25% of the time on random guesses, and writing correct Python is expensive. REINFORCE found the cheap path.

### The fix: shaped reward

The fix is simple in principle — **price tool use explicitly**:
- `+0.3` bonus when the model writes real Python (code extracted, not just wrapped text)
- `+0.2` bonus when the code references opponent stats (`VPIP`, `aggression`)
- `-0.2` penalty when the model falls back to one-word guesses

Result: `real_code` jumps from 0/4 → 3–4/4 per batch, and held-out accuracy improves from 20.0% → 33.3% (+13 pp).

### Layer 2 reward hacking

After the fix, the policy found a new shortcut: write `print("fold")` inside a proper code fence. This earns the code bonus without any reasoning. The natural next reward signal is `exec_ok` — bonus for code that actually **runs** — which would force the policy to write syntactically correct Python.

In [ ]:
# Visualize the code-usage diagnostic across experiments
import matplotlib.pyplot as plt
import numpy as np

experiments = ['Exp B\n(unshaped)', 'Fast Mixed\n(unshaped)', 'Shaped\nReward v1', 'Shaped\nReward v4']
real_code    = [0.0,  0.0,  0.55, 0.95]  # fraction of batch with real code
wrapped      = [1.0,  1.0,  0.45, 0.05]  # fraction that fell back to wrapped

x = np.arange(len(experiments))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, real_code, width, label='Real code (model wrote Python)', color='#27ae60', alpha=0.85)
b2 = ax.bar(x + width/2, wrapped,   width, label='Wrapped/fallback (one-word action)', color='#e74c3c', alpha=0.85)

ax.set_ylabel('Fraction of batch', fontsize=12)
ax.set_title('Code Usage Across Experiments', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(experiments, fontsize=10)
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

for bar in b1:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in b2:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/code_usage_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/code_usage_diagnostic.png')

## 8. Live play vs Slumbot

[Slumbot](https://slumbot.com) is a near-Nash equilibrium no-limit hold'em bot (the 2018 ACPC heads-up no-limit champion) with a free public REST API. Below we put the loaded RL checkpoint up against it for 5 hands — a real-world, opponent-independent benchmark, unlike our heuristic-bounded evaluation above.

**How the bridge works.** Each decision goes through the trained pipeline: the LLM emits Python code, the sandboxed REPL runs it, the last printed line is parsed as `fold / check / call $X / raise $X`, then we translate the action into Slumbot's wire protocol (`f`, `k`, `c`, `b<amount>`) with safe min-bet / min-raise sizing. The cell is **safely skippable** — if no credentials are present it prints a notice and does nothing, so the rest of the notebook still runs.

**Setup.** Register a free account at [slumbot.com](https://slumbot.com), then add `SLUMBOT_USER` and `SLUMBOT_PASS` to the **Colab Secrets** panel (key icon in the left sidebar) so the credentials never enter the notebook history.

> **Honest framing — distribution shift.** The model was trained on 6-max poker at $1/$2 blinds; Slumbot is heads-up at $50/$100 with 200bb stacks. Expect the model to underperform its in-distribution accuracy here. The point of this section is to demonstrate the **integration** — that the RLM pattern (LLM → code → REPL → action) plugs cleanly into a real poker-server protocol — not to claim the model beats Slumbot. The Q-learning agent in the [Range sister project](https://github.com/asrinivasan75/Range) currently sits at **#3 worldwide** on Slumbot's leaderboard at +96.34 BB/100, validating that Slumbot is a non-trivial opponent.


In [ ]:
# _SECTION8_MARKER — Slumbot HTTP client + action-string state parser.
# Self-contained for Colab; mirrors Range/packages/solver/slumbot.py.

import httpx
import re

SLUMBOT_URL = "https://slumbot.com/slumbot"
SB, BB, STACK = 50, 100, 20000


class SlumbotSession:
    """Minimal Slumbot REST client (login / new_hand / act)."""

    def __init__(self, username: str, password: str):
        self.username = username
        self.password = password
        self.token = ""
        self.hands_played = 0
        self.total_winnings = 0

    def login(self) -> bool:
        r = httpx.post(
            f"{SLUMBOT_URL}/api/login",
            json={"username": self.username, "password": self.password},
            headers={"Content-Type": "application/json"},
            timeout=30.0,
        )
        if r.status_code != 200:
            return False
        self.token = r.json().get("token", "")
        return bool(self.token)

    def new_hand(self) -> dict:
        body = {"token": self.token} if self.token else {}
        r = httpx.post(f"{SLUMBOT_URL}/api/new_hand", json=body, timeout=30.0)
        data = r.json()
        if "token" in data:
            self.token = data["token"]
        return data

    def act(self, incr: str) -> dict:
        r = httpx.post(
            f"{SLUMBOT_URL}/api/act",
            json={"token": self.token, "incr": incr},
            timeout=30.0,
        )
        data = r.json()
        if "token" in data:
            self.token = data["token"]
        return data


def _compute_state(action_str: str, client_pos: int) -> dict:
    """Walk Slumbot's action string to derive pot, to_call, and stack state.

    Slumbot convention: position 0 = BTN/SB, position 1 = BB. The action string
    is split per street by '/'. Preflop, position 0 acts first; postflop,
    position 1 (BB) acts first.

    Returns: street, pot, our_invested, opp_invested, to_call, our_remaining,
    our_street_bet, opp_street_bet (the last two are this-street bet totals,
    needed to compute legal Slumbot 'b<amt>' min-raise values).
    """
    streets = action_str.split("/") if action_str else [""]
    street_idx = len(streets) - 1
    total_invested = [SB, BB]  # [pos0=BTN/SB, pos1=BB]
    last_street_bets = [SB, BB]  # captured from the most recently parsed street

    for s_idx, s_actions in enumerate(streets):
        street_bets = [SB, BB] if s_idx == 0 else [0, 0]
        first_actor_pos = 0 if s_idx == 0 else 1
        action_idx = 0
        i = 0
        while i < len(s_actions):
            c = s_actions[i]
            if c == "b":
                j = i + 1
                while j < len(s_actions) and s_actions[j].isdigit():
                    j += 1
                amt = int(s_actions[i + 1 : j])
                who = (first_actor_pos + action_idx) % 2
                total_invested[who] += max(amt - street_bets[who], 0)
                street_bets[who] = amt
                action_idx += 1
                i = j
            elif c == "c":
                who = (first_actor_pos + action_idx) % 2
                top = max(street_bets)
                total_invested[who] += max(top - street_bets[who], 0)
                street_bets[who] = top
                action_idx += 1
                i += 1
            elif c in "kf":
                action_idx += 1
                i += 1
            else:
                i += 1
        last_street_bets = street_bets

    if client_pos == 0:
        our_inv, opp_inv = total_invested[1], total_invested[0]
        our_sb, opp_sb = last_street_bets[1], last_street_bets[0]
    else:
        our_inv, opp_inv = total_invested[0], total_invested[1]
        our_sb, opp_sb = last_street_bets[0], last_street_bets[1]

    return {
        "street": street_idx,
        "pot": our_inv + opp_inv,
        "our_invested": our_inv,
        "opp_invested": opp_inv,
        "to_call": max(0, opp_inv - our_inv),
        "our_remaining": STACK - our_inv,
        "our_street_bet": our_sb,
        "opp_street_bet": opp_sb,
    }


print("Slumbot client + state parser ready.")


In [ ]:
# RLM <-> Slumbot bridge: context builder + action converter.

STREET_NAMES = ["Preflop", "Flop", "Turn", "River"]


def slumbot_to_rlm_context(
    state: dict,
    hole_cards: list,
    board: list,
    client_pos: int,
    action_str: str,
    hand_num: int,
) -> str:
    """Build a prompt in the format the RLM was trained on (mirrors
    src/poker/environment.py:GameState.format_context).

    The trained model expects 6-max with $1/$2 blinds; here we're feeding
    heads-up at $50/$100. Field names and overall shape match; numeric scales
    differ — this is the headline distribution-shift risk for this section.
    """
    position = "BB" if client_pos == 0 else "BTN"
    opp_position = "BTN" if client_pos == 0 else "BB"
    opp_stack = STACK - state["opp_invested"]
    street_name = STREET_NAMES[min(state["street"], 3)]

    lines = [
        "=== POKER HAND ===",
        "Table: Heads-up No-Limit Hold'em",
        f"Blinds: ${SB}/${BB}",
        "",
        f"Your Position: {position}",
        f"Your Stack: ${state['our_remaining']:.0f}",
        f"Your Hand: {hole_cards[0]} {hole_cards[1]}",
        "",
        "Opponents:",
        f"  {opp_position} (Slumbot): ${opp_stack:.0f} (active)",
        "",
    ]
    if board:
        lines.append(f"Community Cards: {' '.join(board)} ({street_name})")
    else:
        lines.append("Street: Preflop (no community cards)")
    lines += ["", f"Pot: ${state['pot']:.0f}"]
    if state["to_call"] > 0:
        pot_odds = state["to_call"] / (state["pot"] + state["to_call"]) * 100
        lines += [f"To Call: ${state['to_call']:.0f}", f"Pot Odds: {pot_odds:.1f}%"]
    lines.append("")
    if action_str:
        lines += [
            "Betting History (this hand, Slumbot wire format):",
            f"  {action_str}",
            "",
        ]
    return "\n".join(lines)


def _parse_rlm_action(text: str):
    """Robustly map RLM stdout to (action_type, amount). Tolerates noise."""
    lines = (text or "").strip().lower().splitlines()
    last = lines[-1].strip() if lines else ""
    if last.startswith("fold"):
        return ("fold", 0.0)
    if last.startswith("check"):
        return ("check", 0.0)
    m = re.match(r"(call|raise|bet)\s*\$?\s*(\d+\.?\d*)", last)
    if m:
        kind = "raise" if m.group(1) == "bet" else m.group(1)
        return (kind, float(m.group(2)))
    for kw in ("fold", "check", "call", "raise"):
        if kw in last:
            return (kw, 0.0)
    return ("fold", 0.0)


def rlm_to_slumbot_incr(predicted: str, state: dict) -> str:
    """Translate the RLM's parsed action into a Slumbot wire token.

    Sizing strategy: ignore the RLM's amount (trained on $1/$2-scale chips,
    not Slumbot's 50/100) and always min-bet / min-raise. Conservative but
    always legal; keeps the demo honest about distribution shift.
    """
    action_type, _amt = _parse_rlm_action(predicted)
    to_call = state["to_call"]
    street = state["street"]
    our_remaining = state["our_remaining"]
    our_street_bet = state["our_street_bet"]
    opp_street_bet = state["opp_street_bet"]

    # Slumbot quirk: preflop the BB option is 'c' (call/check), never 'k'.
    check_token = "k" if street > 0 else "c"

    if action_type == "fold":
        return "f" if to_call > 0 else check_token

    if action_type in ("check", "call"):
        return "c" if to_call > 0 else check_token

    if action_type == "raise":
        if to_call > 0:
            # Min raise (this-street): >= 2x opp's street bet, or +BB.
            target = max(opp_street_bet * 2, opp_street_bet + BB)
        else:
            # No bet to face: open-bet at BB on this street.
            target = max(BB, opp_street_bet + BB)
        # Clamp to all-in: max total this-street bet = our_street_bet + remaining.
        max_legal = our_street_bet + our_remaining
        target = min(target, max_legal)
        if target <= opp_street_bet:
            return "c" if to_call > 0 else check_token
        return f"b{target}"

    return "f" if to_call > 0 else check_token


print("RLM <-> Slumbot bridge ready.")


In [ ]:
# Play loop: one hand at a time, capturing the reasoning trace.

import time


def play_one_hand_vs_slumbot(session, llm_agent, verbose=True, capture_trace=False):
    """Play exactly one hand vs Slumbot using the loaded RLM. Returns a dict."""
    data = session.new_hand()
    if "error" in data:
        if verbose:
            print(f"  Error: {data['error']}")
        return None

    client_pos = data.get("client_pos", 0)
    hole_cards = data.get("hole_cards", [])
    position = "BB" if client_pos == 0 else "BTN"
    hand_num = session.hands_played + 1

    if verbose:
        print(f"\n--- Hand #{hand_num}: {' '.join(hole_cards)} ({position}) ---")

    initial_action = data.get("action", "")
    if initial_action.endswith("f"):
        # Slumbot folded preflop from BTN — we win the SB.
        session.hands_played += 1
        session.total_winnings += SB
        if verbose:
            print(f"  Slumbot folded preflop. +{SB / BB:.1f} bb")
        return {
            "hand_number": hand_num,
            "winnings_bb": SB / BB,
            "cards": hole_cards,
            "final_action": initial_action,
        }

    traces = []
    is_first_response = True
    for turn in range(20):
        action_str = data.get("action", "")
        board = data.get("board", []) or []
        winnings = data.get("winnings")

        if not is_first_response and winnings is not None:
            break
        is_first_response = False

        state = _compute_state(action_str, client_pos)
        context = slumbot_to_rlm_context(
            state, hole_cards, board, client_pos, action_str, hand_num,
        )

        predicted, transcript = llm_agent.run_episode(
            haystack=context,
            question="What should you do?",
            correct_answer="fold",  # unused; placeholder for signature
        )

        incr = rlm_to_slumbot_incr(predicted, state)

        if verbose:
            board_str = " ".join(board) if board else "—"
            print(
                f"  T{turn + 1}: pot=${state['pot']} to_call=${state['to_call']} "
                f"board=[{board_str}]  RLM='{predicted}' -> {incr!r}"
            )

        if capture_trace:
            traces.append({
                "turn": turn + 1,
                "context": context,
                "state": state,
                "predicted": predicted,
                "incr": incr,
                "transcript": transcript,
            })

        data = session.act(incr)

    winnings = data.get("winnings", 0) or 0
    session.hands_played += 1
    session.total_winnings += winnings
    if verbose:
        w = winnings / BB
        print(f"  Result: {'+' if w >= 0 else ''}{w:.1f} bb  |  final action: {data.get('action', '')}")

    return {
        "hand_number": hand_num,
        "winnings_bb": winnings / BB,
        "cards": hole_cards,
        "final_action": data.get("action", ""),
        "final_board": data.get("board", []),
        "traces": traces if capture_trace else None,
    }


def play_vs_slumbot(user: str, password: str, llm_agent, n_hands: int = 5):
    """Login to Slumbot, play n_hands using the RLM, print a tidy summary."""
    session = SlumbotSession(user, password)
    if not session.login():
        print("Login failed. Verify credentials at https://slumbot.com")
        return None
    print(f"Logged in to Slumbot as '{user}'. Playing {n_hands} hands (200bb, 50/100)...")

    results = []
    for i in range(n_hands):
        capture = True  # capture every hand; size is small (<5 hands, few KB each)
        try:
            r = play_one_hand_vs_slumbot(session, llm_agent, verbose=True, capture_trace=capture)
            if r is not None:
                results.append(r)
            time.sleep(0.2)  # be polite
        except Exception as e:
            print(f"  Hand {i + 1} failed: {e!r}")
            time.sleep(1.0)

    n = max(session.hands_played, 1)
    total_bb = session.total_winnings / BB
    wins = sum(1 for r in results if r.get("winnings_bb", 0) > 0)
    print()
    print("=" * 56)
    print(f"  RLM vs Slumbot - {session.hands_played} hands")
    print("=" * 56)
    print(f"  Total:   {'+' if total_bb >= 0 else ''}{total_bb:.1f} bb")
    print(f"  Rate:    {total_bb / n * 100:+.1f} bb/100")
    print(f"  Wins:    {wins}/{session.hands_played}")
    return results


print("Play loop ready: call play_vs_slumbot(user, pw, llm_agent, n_hands=5).")


In [ ]:
# Pull credentials from Colab Secrets (or env vars) and run.

SLUMBOT_USER, SLUMBOT_PASS = "", ""
try:
    from google.colab import userdata
    try:
        SLUMBOT_USER = userdata.get("SLUMBOT_USER") or ""
        SLUMBOT_PASS = userdata.get("SLUMBOT_PASS") or ""
    except Exception:
        pass
except ImportError:
    import os
    SLUMBOT_USER = os.environ.get("SLUMBOT_USER", "")
    SLUMBOT_PASS = os.environ.get("SLUMBOT_PASS", "")

slumbot_results = None
if not SLUMBOT_USER or not SLUMBOT_PASS:
    print("No Slumbot credentials found - skipping live play.")
    print("To enable: add SLUMBOT_USER and SLUMBOT_PASS in the Colab Secrets sidebar.")
    print("Register at https://slumbot.com (free).")
elif "llm_agent" not in dir():
    print("llm_agent is not defined - run sections 3 and 4 above first.")
else:
    slumbot_results = play_vs_slumbot(SLUMBOT_USER, SLUMBOT_PASS, llm_agent, n_hands=5)

    if slumbot_results:
        # Show the first hand that actually produced decisions (Slumbot-fold-preflop hands have no trace).
        hand_with_trace = next((r for r in slumbot_results if r.get("traces")), None)
        if hand_with_trace:
            print()
            print("-" * 60)
            print(f"REASONING TRACE - Hand {hand_with_trace['hand_number']}, Turn 1")
            print("-" * 60)
            t = hand_with_trace["traces"][0]
            print(f"\n[CONTEXT shown to the RLM]\n{t['context'][:1400]}")
            print(f"\n[Parsed RLM action]   {t['predicted']!r}")
            print(f"[Slumbot wire token]  {t['incr']!r}")
            print(f"\n[REPL transcript - {len(t['transcript'])} steps]")
            for step in t["transcript"]:
                print(f"\n--- Step {step.get('step', '?')}: {step.get('action', '')} ---")
                code_src = step.get("code", "")
                if code_src:
                    print(f"Code:\n{code_src[:600]}")
                er = step.get("exec_result", {})
                if er.get("stdout"):
                    print(f"Stdout:\n{er['stdout'][:300]}")
                if er.get("stderr"):
                    print(f"Stderr:\n{er['stderr'][:200]}")


        # _SECTION8_FIGURE_MARKER — also render the captured Slumbot turn visually.
        if hand_with_trace and "render_hand_state" in dir():
            t = hand_with_trace["traces"][0]
            st = t["state"]
            client_pos = 0 if "(BB)" in t["context"] else 1
            board_strs = []
            board_match = re.search(r"Community Cards:\s*([^(]+?)\s*\(", t["context"])
            if board_match:
                board_strs = board_match.group(1).split()
            hole_match = re.search(r"Your Hand:\s*(\S+)\s+(\S+)", t["context"])
            hole = list(hole_match.groups()) if hole_match else ["??", "??"]
            position = "BB" if client_pos == 0 else "BTN"

            fig = render_hand_state(
                hole_cards=hole,
                board=board_strs,
                pot=st["pot"],
                to_call=st["to_call"],
                our_stack=st["our_remaining"],
                opp_stack=STACK - st["opp_invested"],
                position=position,
                opp_label="Slumbot",
                predicted_action=t["predicted"],
                correct_action=None,
                title=f"§8 — Hand {hand_with_trace['hand_number']}, turn {t['turn']} vs Slumbot",
                action_history=None,
            )
            fig.savefig(f"figures/rlm_vs_slumbot_hand{hand_with_trace['hand_number']}.png",
                        dpi=120, bbox_inches="tight")
            plt.show()


## Appendix — Reproducing the Full Training Pipeline

This notebook only runs inference. Full BC + REINFORCE training requires an Ampere+ GPU (A100, L40, H100) and ~90 minutes.

### Quick reference (on a PrimeIntellect H100 pod)

```bash
# 1. Create pod
prime pods create --id <H100-availability-id> --env HF_TOKEN=$HF_TOKEN --name stat4830
prime pods ssh <pod-id>

# 2. Bootstrap (installs Python env, clones repo, unsloth)
INSTALL_UNSLOTH=1 bash <(curl -sSL https://raw.githubusercontent.com/aryanarora236/STAT-4830-RL-project/main/scripts/primeintellect/bootstrap.sh)

# 3. Run the full pipeline (BC + 130 iters RL + eval)
cd /root/STAT-4830-RL-project && source .venv/bin/activate
bash scripts/primeintellect/run_full.sh

# 4. Retrieve artifacts
scp -r pod:/root/STAT-4830-RL-project/checkpoints ./
scp -r pod:/root/STAT-4830-RL-project/docs/results ./docs/
```

Full playbook with troubleshooting at [`scripts/primeintellect/README.md`](../scripts/primeintellect/README.md).

### Shaped-reward run (Modal)

```bash
modal run scripts/modal_shaped_reward.py
```

Results land in `experiments/results/modal_shaped_leaderboard_v1det.json`.